In [16]:
import random
import numpy as np
import pandas as pd
!pip install deap #tem q baixar ele
from deap import base, creator, tools, algorithms
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
import nltk
from nltk.corpus import stopwords

# biblioteca de arquivos do Colab
try:
    from google.colab import files
except ImportError:
    print("Google Colab 'files' não disponível. Assumindo que o arquivo está local.")
    files = None

In [17]:
# baixa e carrega as stopwords
try:
    nltk.download('stopwords')
except Exception as e:
    print(f"Erro ao baixar 'stopwords' do NLTK: {e}")

stopwords = stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [18]:
try:
    # tenta usar o upload do Colab
    print("Por favor, envie o arquivo")
    uploaded = files.upload()
    csv_file = list(uploaded.keys())[0] # pega o nome do primeiro arquivo enviado
    print("Arquivo enviado:", csv_file)
except Exception:
    csv_file = "reviews.csv"
    print(f"Upload não detectado. Tentando carregar '{csv_file}' localmente.")

Por favor, envie o arquivo


Saving brawhalla.csv to brawhalla (2).csv
Arquivo enviado: brawhalla (2).csv


In [19]:
try:
    df = pd.read_csv(csv_file)
    print(f"Arquivo '{csv_file}' carregado com sucesso.")
except FileNotFoundError:
    print(f"ERRO: Arquivo '{csv_file}' não encontrado.")
    raise

Arquivo 'brawhalla (2).csv' carregado com sucesso.


In [20]:
#remove reviews vazias
df.dropna(subset=['review'], inplace=True)

#definição dos documentos e rótulos
docs = df['review'].values.astype('U') # unicode
labels = df['voted_up'].map({True: 1, False: 0}).values

print(f"Carregados {len(docs)} reviews do arquivo.")

Carregados 10364 reviews do arquivo.


In [21]:
X_train_texts, X_test_texts, y_train, y_test = train_test_split(docs, labels, test_size=0.2, random_state=42)

# Cria-se a lista de palavras (bag of words)
vectorizer = CountVectorizer(max_features=4000, stop_words=stopwords)

X_train = vectorizer.fit_transform(X_train_texts)
X_test = vectorizer.transform(X_test_texts)
num_features = X_train.shape[1]
print(f"Vetorizado com {num_features} features.")

Vetorizado com 4000 features.


In [22]:
#Estrutura do Algoritmo Genético
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", lambda: random.randint(0, 1))
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=num_features)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [23]:
# variáveis Globais para salvar o resultado do melhor modelo
best_clf = None
best_mask = None
best_preds = None

# função de avaliação (salva o melhor modelo)
def evaluate(individual, save_model=False):
    global best_clf, best_mask, best_preds

    mask = np.array(individual, dtype=bool)
    if mask.sum() == 0:
        return (0.0,)

    X_train_sel = X_train[:, mask]
    X_test_sel = X_test[:, mask]

    clf = MultinomialNB()
    clf.fit(X_train_sel, y_train)
    preds = clf.predict(X_test_sel)
    acc = accuracy_score(y_test, preds)

    # salva o resultado se for a chamada final
    if save_model:
        best_clf = clf
        best_mask = mask
        best_preds = preds

    return (acc,)

In [24]:
# operadores genéticos
toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

In [25]:
# parametros do GA
pop = toolbox.population(n=20)
NGEN = 25
CXPB, MUTPB = 0.5, 0.2

In [28]:

# execução
print("Iniciando Algoritmo Genético...")
for gen in range(NGEN):
    offspring = algorithms.varAnd(pop, toolbox, cxpb=CXPB, mutpb=MUTPB)
    fits = list(map(toolbox.evaluate, offspring))

    for fit, ind in zip(fits, offspring):
        ind.fitness.values = fit

    pop = toolbox.select(offspring, k=len(pop))

print("GA finalizado.")

best_ind = tools.selBest(pop, k=1)[0]
# chamada final da avaliação, pedindo para salvar o modelo e as previsões
best_acc = evaluate(best_ind, save_model=True)[0]

# Resultados e Visualização

print(f"\n Resultados Finais para {csv_file}")
print(f"Melhor acurácia: {best_acc:.4f}")
print(f"Features selecionadas: {np.sum(best_ind)} / {num_features}")

# mapeamento
STATUS = {1: "POSITIVO (1)", 0: "NEGATIVO (0)"}

# Separa índices por rótulo real (y_test)
indices_pos = np.where(y_test == 1)[0]
indices_neg = np.where(y_test == 0)[0]

# seleciona 5 de cada, se tiver é claro
num_samples = 5
sample_pos = random.sample(list(indices_pos), min(num_samples, len(indices_pos)))
sample_neg = random.sample(list(indices_neg), min(num_samples, len(indices_neg)))

sample_indices = sample_pos + sample_neg # Combina as amostras

print("\n AMOSTRAS DE PREDIÇÃO (5 Positivas Reais e 5 Negativas Reais) ")

for i, idx in enumerate(sample_indices):
    real_label = y_test[idx]
    predicted_label = best_preds[idx]
    review_text = X_test_texts[idx].replace('\n', ' ').strip() # limpa quebras de linha

    print(f"\n[{i+1}]")
    print(f"  > Rótulo Real:        {STATUS[real_label]}")
    print(f"  > Predição do Modelo: {STATUS[predicted_label]}")

    # indica se o modelo acertou
    acertou = " Acerto" if predicted_label == real_label else " Erro"
    print(f"  > Status:             {acertou}")

    # limita o texto para melhor visualização adicione isso após o review_text se qser limitar os textos[:150]
    print(f"  > Review:             {review_text}")

Iniciando Algoritmo Genético...
GA finalizado.

 Resultados Finais para brawhalla (2).csv
Melhor acurácia: 0.7689
Features selecionadas: 2036 / 4000

 AMOSTRAS DE PREDIÇÃO (5 Positivas Reais e 5 Negativas Reais) 

[1]
  > Rótulo Real:        POSITIVO (1)
  > Predição do Modelo: POSITIVO (1)
  > Status:              Acerto
  > Review:             Can pound men and still be straight. 10/10

[2]
  > Rótulo Real:        POSITIVO (1)
  > Predição do Modelo: POSITIVO (1)
  > Status:              Acerto
  > Review:             gg geming gais

[3]
  > Rótulo Real:        POSITIVO (1)
  > Predição do Modelo: POSITIVO (1)
  > Status:              Acerto
  > Review:             FUN!!! play now

[4]
  > Rótulo Real:        POSITIVO (1)
  > Predição do Modelo: POSITIVO (1)
  > Status:              Acerto
  > Review:             it's nice

[5]
  > Rótulo Real:        POSITIVO (1)
  > Predição do Modelo: POSITIVO (1)
  > Status:              Acerto
  > Review:             cool game

[6]
  > Rótulo Re